In [ ]:
from matplotlib import pyplot as plt
from natsort import natsorted
from PIL import Image, ImageEnhance
from readlif.reader import LifFile
from skimage import color
from skimage import exposure
from skimage import io
from stapl3d.preprocessing import shading
import argparse
import cv2
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import os
import random
import read_lif
import shutil
import tifffile

### Funciones

In [ ]:
# Function increases/decreases randomly the brightness of the passed imaged based on a distribution of values passed as a list
def BrightnessAugmentation(image,brightnesslist):
    image = np.array(image)
    brightness = random.sample(brightnesslist, 1)
    print(brightness)
    # print(brightness)
    pilOutput = Image.fromarray(image)
    enhancer = ImageEnhance.Brightness(pilOutput)
    output = enhancer.enhance(brightness[0])

    augmented_image = np.array(output)

    return(augmented_image)

In [ ]:
#FOR TRAINING AND TESTING A MODEL
#Function extracts patch, normalizes it based on percentile value of each channel, and creates syntheic data, returning concatenated AB images representing the source and target image.
def get_patch_train(l, uby, ubx, patchsize, channels, channel_stacks, percentiles, mode, alpha, normalization, Brightness):
    
    # Generate brightness range list for data augmentation
    brightnessRange = [0.5, 3]
    l_br = int(brightnessRange[0] * 100)
    u_br = int(brightnessRange[1] * 100)
    brightnesslist = [round(x * 0.01, 2) for x in range(l_br, u_br, 1)]

    # Setting arguments to integers
    uby = int(uby)
    ubx = int(ubx)

    # Alpha value for membrane channel
    if alpha:
        alpha2 = round(1 - alpha, 2)

    # Getting coordinates for patch extraction
    i_idx = random.randint(0, uby)
    j_idx = random.randint(0, ubx)
    yax = [i_idx, (i_idx + patchsize)]
    xax = [j_idx, (j_idx + patchsize)]

    # Generates a random number representing a random selection of a layer
    rl = random.randint(0, l - 1)

    # List containing normalized patches from all three channels
    channel_normalized_patches_list = []

    # For each channel, get the patch using the indices in the yax and xax lists
    for c in range(len(channels)):
        # Extracting the patch from channel "c" from the layer "rl" of the stack
        patch = channel_stacks[c][rl, yax[0]:yax[1], xax[0]:xax[1]]

        if normalization == "ac":
            # Normalizing the patch using the percentile of the current channel "c"
            normalized_patch = (patch / percentiles[c]) * 255

        if normalization == "od":
            # Normalizing the patch using the highest percentile (open detector channel percentile)
            normalized_patch = (patch / percentiles[2]) * 255

        # Clipping the values, anything higher than 255 is set to 255
        normalized_patch[normalized_patch > 255] = 255
        # Adding the normalized patch of current channel "c" to the list which collects all 3 channels
        channel_normalized_patches_list.append(normalized_patch)

    # Empty 3 channel RGB array for source image
    source_image = np.zeros((patchsize, patchsize, 3))

    if mode == "synthetic":
        # Creating the synthetic patch by taking the mean value of the nuclear and membrane channel
        synthetic_patch = (channel_normalized_patches_list[0] + channel_normalized_patches_list[1]) / 2
        synthetic_patch = synthetic_patch.astype(np.uint8)

        # Introducing the Synthetic patch in each channel of the source image (RGB)
        source_image[:, :, 0] = synthetic_patch.astype(np.uint8)
        source_image[:, :, 1] = synthetic_patch.astype(np.uint8)
        source_image[:, :, 2] = synthetic_patch.astype(np.uint8)

    elif mode == "weighted":
        # Creating the synthetic weighted blended patch by taking the mean value of the nuclear and membrane channel blended with alpha and 1-alpha
        synthetic_patch = cv2.addWeighted(channel_normalized_patches_list[0], alpha2, channel_normalized_patches_list[1], alpha, 0)
        synthetic_patch = synthetic_patch.astype(np.uint8)

        # Introducing the Synthetic patch in each channel of the source image (RGB)
        source_image[:, :, 0] = synthetic_patch.astype(np.uint8)
        source_image[:, :, 1] = synthetic_patch.astype(np.uint8)
        source_image[:, :, 2] = synthetic_patch.astype(np.uint8)

    elif mode == "open_detector":
        # Introducing the Real patch in each channel of the source image (RGB)
        source_image[:, :, 0] = channel_normalized_patches_list[2].astype(np.uint8)
        source_image[:, :, 1] = channel_normalized_patches_list[2].astype(np.uint8)
        source_image[:, :, 2] = channel_normalized_patches_list[2].astype(np.uint8)

    if Brightness:
        # Enhancing brightness of source image
        if random.randint(0, 100) < int(Brightness):
            source_image = BrightnessAugmentation(source_image.astype(np.uint8), brightnesslist)
        else:
            source_image = source_image.astype(np.uint8)

    # Empty 3 channel RGB array for target image
    target_image = np.zeros((patchsize, patchsize, 3))

    # Introducing the nuclear channel in the Red channel and membrane in the Green channel
    target_image[:, :, 0] = channel_normalized_patches_list[1].astype(np.uint8)
    target_image[:, :, 1] = channel_normalized_patches_list[0].astype(np.uint8)

    # Concatenating the source and the target image to get the AB format for pix2pix
    image_AB = np.concatenate([source_image, target_image], 1)

    # Return the concatenated AB image
    return image_AB

In [ ]:
# Función para crear directorios si no existen
def create_directories(base_dir):
    os.makedirs(base_dir, exist_ok=True)
    for subdir in ["train", "test", "val"]:
        os.makedirs(os.path.join(base_dir, subdir), exist_ok=True)

In [ ]:
# Función para calcular percentiles para cada canal
def calculate_percentiles(filepaths, l_layer, u_layer, percentile):
    channel_stacks = []
    percentiles = []
    
    for filepath in filepaths:
        tmp_channel_stacked_planes = []
        for layer in range(l_layer, u_layer):
            data = tifffile.imread(filepath)[layer, :, :]
            tmp_channel_stacked_planes.append(data)
        
        planes_stacked = np.stack(tmp_channel_stacked_planes, axis=0)
        channel_stacks.append(planes_stacked)
        flat_stacked_planes = planes_stacked.flatten()
        p_val = np.percentile(flat_stacked_planes, percentile)
        percentiles.append(p_val)
    
    return channel_stacks, percentiles

In [ ]:
# Función para extraer y guardar parches de entrenamiento, prueba y validación
def extract_and_save_patches(datafoldername, t_train, t_test, t_val, patchsize, channels, channel_stacks, percentiles, mode, alpha, Norm, Brightness):
    dimensions = channel_stacks[0].shape
    l = dimensions[0]
    y = dimensions[1]
    x = dimensions[2]
    ubx = x - patchsize
    uby = y - patchsize

    for phase, count in zip(['train', 'test', 'val'], [t_train, t_test, t_val]):
        for t in range(count):
            image_AB = get_patch_train(l, uby, ubx, patchsize, channels, channel_stacks, percentiles, mode, alpha, Norm, Brightness)
            filename = f"{datafoldername}/{phase}/{t}.png"
            matplotlib.image.imsave(filename, image_AB.astype(np.uint8))
    

In [ ]:
image1 = tifffile.imread(ruta_tiff1)
image2 = tifffile.imread(ruta_tiff2)
image3 = tifffile.imread(ruta_tiff3)

In [ ]:
image1.shape

In [ ]:
image2.shape

In [ ]:
image3.shape

In [ ]:
# Verificar que todas las imágenes tengan las mismas dimensiones
assert image1.shape == image2.shape == image3.shape, "Las dimensiones de las imágenes no coinciden"

In [ ]:
depth, alto, ancho = image1.shape

In [ ]:
depth

In [ ]:
# Crear una imagen vacía con tres canales
combined_image = np.zeros((depth, alto, ancho, 3), dtype=image1.dtype)

In [ ]:
combined_image.shape

In [ ]:
# Asignar cada imagen a un canal del array combinado
combined_image[..., 0] = image1  # Canal Rojo
combined_image[..., 1] = image2  # Canal Verde
combined_image[..., 2] = image3  # Canal Azul

In [ ]:
# Guardar la imagen combinada como un archivo TIFF con tres canales
tifffile.imwrite('../imagen_combinada_3_canales.tiff', combined_image)

### traintest

In [ ]:
ch1, ch2, ch3 = ruta_tiff1, ruta_tiff2, ruta_tiff3

In [ ]:
# if role == "train_test":
# Lista de filepaths para cada canal
filepaths = [ch1, ch2, ch3]
l_layer, u_layer = 10, 20  # Define tus límites de capas
percentile = 99  # Define tu valor percentil

channel_stacks, percentiles = calculate_percentiles(filepaths, l_layer, u_layer, percentile)

datasize = 100  # Define el tamaño de tus datos
patchsize = 512  # Define el tamaño del parche
alpha = 0.5
Norm = "ac"
mode = "open_detector"
Brightness = "50"
biosample = "lif"

t_train = int(datasize)
t_test = int(t_train * 0.2)
t_val = int(t_train * 0.2)

if alpha:
    wb = str(alpha) + "_" + str(round(1 - alpha, 2))
    datafoldername = f"../{biosample}_{datasize}_{Norm}_{mode}_{wb}_patches"
elif Brightness:
    datafoldername = f"../{biosample}_{datasize}_{Norm}_{mode}_br{Brightness}%_patches"
else:
    datafoldername = f"../{biosample}_{datasize}_{Norm}_{mode}_patches"

create_directories(datafoldername)

extract_and_save_patches(datafoldername, t_train, t_test, t_val, patchsize, filepaths, channel_stacks, percentiles, mode, alpha, Norm, Brightness)

### Funciones

In [ ]:
def process_channel_to_tiff(series, channel_idx, num_frames, ruta_base, sample_name):
    """
    Procesa un canal de una serie y guarda las imágenes en un archivo TIFF multipágina.

    Args:
        series (LifSeries): Serie a procesar.
        channel_idx (int): Índice del canal a procesar.
        num_frames (int): Número de capas a procesar.
        ruta_base (str): Ruta base para guardar los archivos.
        sample_name (str): Nombre del canal.

    Returns:
        str: Ruta al archivo TIFF multipágina creado.
    """
    ruta_carpeta = os.path.join(ruta_base, sample_name)

    if not os.path.exists(ruta_carpeta):
        os.makedirs(ruta_carpeta)

    # Guardar cada capa como un archivo TIFF separado
    for i in range(num_frames):
        chosen = series.get_frame(c=channel_idx, z=i)
        tifffile.imwrite(f'{ruta_carpeta}/{sample_name}_{i}.tiff', chosen)

    # Obtener una lista de todos los archivos TIFF en la carpeta
    tiff_files = [f for f in os.listdir(ruta_carpeta) if f.endswith('.tiff') or f.endswith('.tif')]

    # Ordenar la lista de archivos si es necesario
    tiff_files = natsorted(tiff_files)

    # Leer y almacenar todos los frames de los archivos TIFF
    frames = []
    for tiff_file in tiff_files:
        with tifffile.TiffFile(os.path.join(ruta_carpeta, tiff_file)) as tif:
            for page in tif.pages:
                frames.append(page.asarray())

    # Guardar todos los frames en un solo archivo TIFF multipágina
    tifffile.imwrite(f'{ruta_carpeta}/{sample_name}.tiff', frames)

    # Devuelve la ruta al archivo TIFF multipágina creado
    return os.path.join(ruta_carpeta, f'{sample_name}.tiff')

In [ ]:
def choose_and_process_lif_channel(lif_path, series_idx, channel_idx, num_frames, ruta_base, sample_name):
    """
    Elige una serie y un canal del archivo LIF y los transforma en un archivo TIFF multipágina.

    Args:
        lif_path (str): Ruta al archivo LIF.
        series_idx (int): Índice de la serie a procesar.
        channel_idx (int): Índice del canal a procesar.
        num_frames (int): Número de capas a procesar.
        ruta_base (str): Ruta base para guardar los archivos.
        sample_name (str): Nombre del canal.

    Returns:
        str: Ruta al archivo TIFF multipágina creado.
    """
    reader = LifFile(lif_path)
    series = reader.get_image(series_idx)
    return process_channel_to_tiff(series, channel_idx, num_frames, ruta_base, sample_name)

In [ ]:
def process_multiple_channels(lif_path, series_list, channel_list, num_frames, ruta_base):
    """
    Procesa múltiples series y canales y guarda las imágenes en archivos TIFF multipágina.

    Args:
        lif_path (str): Ruta al archivo LIF.
        series_list (list): Lista de índices de las series a procesar.
        channel_list (list): Lista de índices de los canales a procesar.
        num_frames (int): Número de capas a procesar.
        ruta_base (str): Ruta base para guardar los archivos.
    """
    tiff_paths = []
    for series_idx, channel_idx in zip(series_list, channel_list):
        sample_name = f'channel_{series_idx + 1}_{channel_idx + 1}'
        ruta_tiff = choose_and_process_lif_channel(lif_path, series_idx, channel_idx, num_frames, ruta_base, sample_name)
        tiff_paths.append(ruta_tiff)
    
    return tiff_paths

### Canal 1

In [ ]:
# Elegir serie y canal para transformar
series_idx = 0  # Elige el índice de la serie
channel_idx = 0  # Elige el índice del canal
num_frames = 35
ruta_base = '../canales2'
sample_name = f'channel_{i}'
i = i + 1

In [ ]:
# Procesar la serie y el canal elegidos
choose_and_process_lif_channel(reader, series_idx, channel_idx, num_frames, ruta_base, sample_name)

### Canal 2

In [ ]:
# Elegir serie y canal para transformar
series_idx = 0  # Elige el índice de la serie
channel_idx = 1  # Elige el índice del canal
num_frames = 35
ruta_base = '../canales2'
sample_name = f'channel_{i}'
i = i + 1

In [ ]:
# Procesar la serie y el canal elegidos
choose_and_process_lif_channel(reader, series_idx, channel_idx, num_frames, ruta_base, sample_name)

### Canal 3

In [ ]:
# Elegir serie y canal para transformar
series_idx = 1  # Elige el índice de la serie
channel_idx = 0  # Elige el índice del canal
num_frames = 35
ruta_base = '../canales2'
sample_name = f'channel_{i}'
i = i + 1

In [ ]:
# Procesar la serie y el canal elegidos
choose_and_process_lif_channel(reader, series_idx, channel_idx, num_frames, ruta_base, sample_name)

### FINAL

In [ ]:
# Ejemplo de uso
lif_path = '../data/flim_sirDNA_cellbrite650_zsatck.lif'
series_list = [0, 0, 1]  # Lista de índices de las series a procesar
channel_list = [0, 1, 0]  # Lista de índices de los canales a procesar
num_frames = 35
ruta_base = '../channels'

# Procesar las series y canales especificados
tiff_paths = process_multiple_channels(lif_path, series_list, channel_list, num_frames, ruta_base)

# Imprimir las rutas de los archivos TIFF finales
print("Archivos TIFF finales:")
for path in tiff_paths:
    print(path)

In [ ]:
# if role == "train_test":
# Lista de filepaths para cada canal
filepaths = tiff_paths
l_layer, u_layer = 10, 20  # Define tus límites de capas
percentile = 99  # Define tu valor percentil

channel_stacks, percentiles = calculate_percentiles(filepaths, l_layer, u_layer, percentile)

datasize = 100  # Define el tamaño de tus datos
patchsize = 512  # Define el tamaño del parche
alpha = 0.5
Norm = "ac"
mode = "open_detector"
Brightness = "50"
biosample = "lif_final"

t_train = int(datasize)
t_test = int(t_train * 0.2)
t_val = int(t_train * 0.2)

if alpha:
    wb = str(alpha) + "_" + str(round(1 - alpha, 2))
    datafoldername = f"../{biosample}_{datasize}_{Norm}_{mode}_{wb}_patches"
elif Brightness:
    datafoldername = f"../{biosample}_{datasize}_{Norm}_{mode}_br{Brightness}%_patches"
else:
    datafoldername = f"../{biosample}_{datasize}_{Norm}_{mode}_patches"

create_directories(datafoldername)

extract_and_save_patches(datafoldername, t_train, t_test, t_val, patchsize, filepaths, channel_stacks, percentiles, mode, alpha, Norm, Brightness)
shutil.rmtree(ruta_base)